In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd 
import seaborn as sns
import sys 
import glob
import torch 
import os
from tqdm import tqdm
from torch.utils.data import DataLoader
import gc
import yaml
import h5py
from typing import Any, Union

### defining functions

In [2]:
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/transforms")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/configs")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/plots")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/models")

In [3]:
from fixed_dataset import FixedDataset
from seg_recon_vit3d import SegRecon_ViT_3D
from transforms.factory import transform_factory
from utils.read_yaml import read_yaml
from utils.model_select import model_select
from utils.load_ckpt import load_best_ckpt
from loss import loss_select
from torchmetrics.image import StructuralSimilarityIndexMeasure
from metrics.sam import SAMScore, Intensity_SAMScore
from transforms import *
from transforms import FourierSpectralTransform
from transforms.inverse_factory import *

/home/ana-caznok/software/src/miniforge3/envs/agrvai/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def replace_zeros_with_small_value(tensor: torch.Tensor, small_value: float = 0.0001) -> torch.Tensor:

    # Clone the tensor to avoid modifying the original tensor in-place
    result = tensor.clone()

    # Create a boolean mask where the tensor is zero
    zero_mask = result == 0

    # Replace all values in result where mask is True with the small_value
    result[zero_mask] = small_value

    return result

In [5]:
def plot_channel_differences(y: torch.Tensor, out: torch.Tensor, channels: list,model_name:str, base_path = "/media/ana-caznok/SSD-08/recon-segment/"):
    """restormer_fft2ifft2hsi_noclip.yaml
    Plots the difference between corresponding channels of `y` and `out` for the specified channels.

    Parameters:
    - y (torch.Tensor): Tensor of shape (61, 256, 256)
    - out (torch.Tensor): Tensor of shape (1, 61, 256, 256)
    - channels (list of int): List of channel indices to plot
    """
    # Remove the batch dimension from `out`, now shape (61, 256, 256)
    if y.type() != out.type():
        out = out.detach().cpu()
    out = out.squeeze(0)

    # Verify tensor shapes
    assert y.shape == out.shape == (61, 256, 256), "Shape mismatch: Expected (61, 256, 256)"

    # Compute the difference for each selected channel and store in a list
    differences = [np.abs((y[c] - out[c]).detach().cpu().numpy()) for c in channels]

    # Determine global min and max for consistent color scale
    global_min = 0 #min(diff.min() for diff in differences)
    global_max = 1 #max(diff.max() for diff in differences)

    # Determine subplot layout
    n_channels = len(channels)
    ncols = min(4, n_channels)
    nrows = (n_channels + ncols - 1) // ncols

    # Create figure and axes
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = axes.flatten() if n_channels > 1 else [axes]

    # Plot each channel difference
    for idx, (channel, diff) in enumerate(zip(channels, differences)):
        ax = axes[idx]
        im = ax.imshow(diff, cmap='magma', vmin=global_min, vmax=global_max)
        ax.set_title(f'Channel {channel} Difference')
        ax.axis('off')

    # Turn off unused subplots if any
    for idx in range(len(differences), len(axes)):
        axes[idx].axis('off')

    # Add a common colorbar
    fig.colorbar(im, ax=axes, orientation='vertical', fraction=0.02, pad=0.04)
    fig.suptitle(model_name)

    # Adjust layout to prevent overlap
    #plt.tight_layout()
    plt.savefig(base_path + 'plots/' + 'err_' + model_name + '.png')

    plt.show()

In [6]:
def plot_output_channels(
    out: torch.Tensor,
    channels: list,
    model_name: str,
    base_path: str = "/media/ana-caznok/SSD-08/recon-segment/"):
    
    # Remove batch dimension, resulting shape (C, H, W)
    out = out.squeeze(0)

    # Check if output has correct dimensions
    assert out.ndim == 3, "Expected tensor shape (C, H, W) after squeezing batch dimension"

    # Extract the selected channels and move to CPU
    selected_images = [out[c].detach().cpu().numpy() for c in channels]

    # Determine layout for subplots
    n_channels = len(channels)
    ncols = min(4, n_channels)
    nrows = (n_channels + ncols - 1) // ncols

    # Create subplots
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = axes.flatten() if n_channels > 1 else [axes]

    # Plot each selected channel
    for idx, (channel, img) in enumerate(zip(channels, selected_images)):
        ax = axes[idx]
        im = ax.imshow(img, cmap='gray')
        ax.set_title(f'Channel {channel}')
        ax.axis('off')

    # Turn off unused axes if there are any
    for idx in range(len(selected_images), len(axes)):
        axes[idx].axis('off')

    # Add colorbar for last image
    fig.colorbar(im, ax=axes, orientation='vertical', fraction=0.02, pad=0.04)
    fig.suptitle(f'Output Channels - {model_name}')

    # Save the figure
    plt.savefig(base_path + 'plots/' + 'out_channels_' + model_name + '.png')
    plt.show()


In [7]:
def plot_spectral_input(x, output, y, tp, 
                        model_name, key,
                        points = [(148, 140), (40, 40)], 
                        base_path: str = "/media/ana-caznok/SSD-08/recon-segment/"):
    
    # Squeeze output if it has a batch dimension
    if output.ndim == 4:
        output = output.squeeze(0)
        
    if y.type() != output.type():
       output = output.detach().cpu().numpy()
    # Move tensors to numpy
    y = y.detach().cpu().numpy()
    output = output.detach().cpu().numpy()
    x = x.detach().cpu().numpy()

    if tp =='msi':
        wavl = np.linspace(400,1000,61)
        rgbnir = [420,500,630,960] 
        _,wavl_idx = np.unique(wavl,return_index=True)
        rgbnir_idx = wavl_idx[np.isin(wavl, rgbnir)]
        x_plot = rgbnir_idx
        x_min,x_max = wavl_idx[0], wavl_idx[-1]
        x_domain = 'wavelenght'

    else:
        x_plot = np.arange(0,x.shape[0],1)
        x_min,x_max = 0, x.shape[0]

    # Create 2x2 subplots
    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    input_colors = ['hotpink', 'red']
    my_colors = ['turquoise', 'teal', 'mediumorchid', 'darkorchid']

    # --- TOP LEFT: Middle band of input x ---
    mid_band = x.shape[0] // 2
    axs[0, 0].imshow(x[mid_band, :, :], cmap='gray')
    axs[0, 0].set_title(f"Input Image - Band {mid_band}")

    # Mark selected points
    for i, (row, col) in enumerate(points):
        axs[0, 0].plot(col, row, '*', color=input_colors[i], markersize=10)

    # --- TOP RIGHT: Spectral values from input x ---
    for i, (row, col) in enumerate(points):
        axs[0, 1].plot(x_plot, x[:, row, col],'o', color=input_colors[i], label=f'Input ({row},{col})')

    axs[0, 1].set_title("Spectral Signature of Input (x)")
    axs[0, 1].set_xlabel("Band Index")
    axs[0, 1].set_ylabel("Intensity")
    axs[0,1].set_xlim([x_min,x_max])
    axs[0, 1].legend()

    # --- BOTTOM LEFT: GT image with points ---
    axs[1, 0].imshow(y[30, :, :], cmap='gray')
    axs[1, 0].set_title("GT Band 30 with Points")
    for i, (row, col) in enumerate(points):
        axs[1, 0].plot(col, row, '*', color=my_colors[i*2 + 1], markersize=10)

    # --- BOTTOM RIGHT: GT vs Output Spectra ---
    for i, (row, col) in enumerate(points):
        axs[1, 1].plot(y[:, row, col], color=my_colors[i*2], label=f'GT ({row},{col})')
        axs[1, 1].plot(output[:, row, col], color=my_colors[i*2 + 1], linestyle='-.', label=f'Output ({row},{col})')
    axs[1, 1].set_title("Spectral Comparison (GT vs Output)")
    axs[1, 1].set_xlabel("Band Index")
    axs[1, 1].set_ylabel("Intensity")
    axs[1, 1].legend()

    # Overall figure title and layout adjustment
    fig.suptitle(model_name, fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(base_path + 'plots/in-out_' + model_name + 'png')
    plt.show()


In [8]:
def plot_spectral_diff(y, output, model_name, points=[(148, 140), (40, 40)]):
    """
    Plots spectral differences between ground truth and model output at given spatial points.

    Parameters:
        y (Tensor or ndarray): Ground truth hyperspectral data of shape (bands, H, W).
        output (Tensor or ndarray): Model output data, expected to be (1, bands, H, W) or (bands, H, W).
        model_name (str): Title for the plot.
        points (list of tuple): List of (row, col) coordinates for which to plot spectra.
    """
    
    # Remove batch dimension if present
    if output.ndim == 4:
        output = output.squeeze(0)
    y = y.detach().cpu().numpy()
    output = output.detach().cpu().numpy()
    # Create a figure with two subplots
    fig, axs = plt.subplots(1, 2, figsize=(15, 6))
    
    # Show a grayscale image at an arbitrary band (e.g., 30)
    axs[0].imshow(y[30, :, :], cmap='gray')
    axs[0].set_title("Spatial Locations")
    
    my_colors = ['teal', 'darkorange']
    
    for i, (row, col) in enumerate(points):
        # Mark point on the image
        axs[0].plot(col, row, '*', color=my_colors[i], markersize=10)
        
        # Plot ground truth and output spectra at this point
        axs[1].plot(y[:, row, col], color=my_colors[i], label=f'GT ({row},{col})')
        axs[1].plot(output[:, row, col], color=my_colors[i], linestyle='-.', label=f'Output ({row},{col})')
    
    axs[1].set_title("Spectral Comparison")
    axs[1].set_xlabel("Spectral Band")
    axs[1].set_ylabel("Intensity")
    axs[1].legend()
    
    fig.suptitle(model_name)
    plt.tight_layout()
    plt.show()

In [9]:
def replace_zeros_with_small_value(tensor: torch.Tensor, small_value: float = 0.0001) -> torch.Tensor:

    # Clone the tensor to avoid modifying the original tensor in-place
    result = tensor.clone()

    # Create a boolean mask where the tensor is zero
    zero_mask = result == 0

    # Replace all values in result where mask is True with the small_value
    result[zero_mask] = small_value

    return result

In [10]:
def calc_plot_ssim(y, out, model_name, base_path: str = "/media/ana-caznok/SSD-08/recon-segment/"):

    # Ensure both tensors are on CPU and detached for visualization
    if y.type() != out.type():
        out = out.detach().cpu()

    #y = y.detach().cpu()

    # Compute SAM and Intensity-SAM with corresponding maps
    ssim_func = StructuralSimilarityIndexMeasure()
    channel_nb = y.shape[1]
    ssim = []
    for c in range(channel_nb):
        ssim_c = ssim_func(y[0,c,:,:].unsqueeze(0).unsqueeze(0),out[0,c,:,:].unsqueeze(0).unsqueeze(0)).detach().cpu().numpy()
        ssim.append(ssim_c)

    ssim = np.array(ssim)

    plt.plot(ssim)
    plt.title(model_name)
    plt.ylabel('SSIM')
    plt.xlabel('channel')
    plt.savefig(base_path + 'plots/spectral-ssim_' + model_name + '.png')
    plt.show()

    return ssim

In [11]:
def plot_all_models(data_dict, base_path: str = "/media/ana-caznok/SSD-08/recon-segment/"):
    """
    Plots all arrays in a single figure with each line labeled by its dictionary key.

    Parameters:
    - data_dict (dict): A dictionary where keys are model names and values are 1D arrays.
    """
    plt.figure(figsize=(8, 8))
    
    # Plot each array with the key as legend
    for key, array in data_dict.items():
        plt.plot(array, label=key)

    # Add legend and labels
    plt.legend()
    plt.title('All Models Comparison')
    plt.xlabel('Channels')
    plt.ylabel('SSIM')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(base_path + 'spectral-compare_ssim.png')
    plt.show()


def plot_grouped_models(data_dict, base_path: str = "/media/ana-caznok/SSD-08/recon-segment/", only_vis=False):
    """
    Plots subplots grouped by model types ('restormer' and 'unet').
    Each subplot contains two lines: one for 'msi2hsi' and another for 'fft2ifft2hsi'.

    Parameters:
    - data_dict (dict): A dictionary where keys are model-transformation identifiers and values are 1D arrays.
    """
    # Initialize subplots for each model type
    lbl_dict = {}
    for d in data_dict.keys(): 
        if only_vis: 
            lbl_dict[d] = d
        else: 
            if 'interp31' in d: 
                lbl_dict[d] = 'Contains NIR info ' + d
            elif 'msi' in d: 
                lbl_dict[d] = 'Contains NIR info ' + d
            else:
                lbl_dict[d] = 'Pseudo_hyper from RGB ' + d
    
    fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    
    model_types = ['restormer', 'unet']

    for idx, model in enumerate(model_types):
        ax = axs[idx]
        for key, array in data_dict.items():
            # Check if the key contains the current model and a valid transformation
            if model in key and ('msi2hsi' in key or 'fft2ifft2hsi' in key or 'rgb' in key) :
                ax.plot(array, label=lbl_dict[key])
        
        # Set title and labels for the subplot
        ax.set_title(f'{model.upper()} Models')
        ax.set_xlabel('Channels')
        if idx == 0:
            ax.set_ylabel('SSIM')
        ax.grid(True)
        ax.legend()

    plt.suptitle('Model Comparison by Transformation Type')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(base_path + 'spectral_compare_ntire_skin.png')
    plt.show()

In [12]:
def calc_sam(y,out): 
    if y.type() != out.type():
        out = out.detach().cpu()

    # Compute SAM and Intensity-SAM with corresponding maps
    sam_func = SAMScore()
    isam_func = Intensity_SAMScore()

    sam, sam_map = sam_func(y, out, return_maps=True)
    isam, isam_map = isam_func(y, out, return_maps=True)

    sam_map = sam_map.detach().cpu().numpy()
    isam_map = isam_map.detach().cpu().numpy()
    sam = float(sam.detach().cpu().numpy())
    isam = float(isam.detach().cpu().numpy())

    return sam, isam


In [13]:
def calc_ssim(y,out): 
    if y.type() != out.type():
        out = out.detach().cpu()

    ssim_func = StructuralSimilarityIndexMeasure()
    channel_nb = y.shape[1]
    channel_ssim = []
    for c in range(channel_nb):
        ssim_c = ssim_func(y[0,c,:,:].unsqueeze(0).unsqueeze(0),out[0,c,:,:].unsqueeze(0).unsqueeze(0)).detach().cpu().numpy()
        channel_ssim.append(ssim_c)

    channel_ssim = np.array(channel_ssim)
    ssim = np.mean(channel_ssim)

    return ssim, channel_ssim

In [14]:
def calc_error(y,out): 
    if y.type() != out.type():
        out = out.detach().cpu()

    error = torch.abs(y - out)
    error = float(torch.mean(error).detach().cpu().numpy())

    return error

In [15]:
def calc_plot_sam(y, out, m, model_name, base_path: str = "/media/ana-caznok/SSD-08/recon-segment/"):
    """
    Calculate and plot SAM and Intensity-SAM score maps side-by-side with a shared intensity scale bar.
    """

    # Ensure both tensors are on CPU and detached for visualization
    if y.type() != out.type():
        out = out.detach().cpu()
        #mask = mask.cpu().detach()

    #y = y.detach().cpu()
    mask = m['mask'].unsqueeze(0)
    mask = mask.cpu().detach()
    masked_y = replace_zeros_with_small_value(mask*y,0.001)
    masked_out = replace_zeros_with_small_value(mask*out,0.002)
    print(masked_out.shape)
    # Compute SAM and Intensity-SAM with corresponding maps
    sam_func = SAMScore()
    isam_func = Intensity_SAMScore()

    sam, sam_map = sam_func(y, out, return_maps=True)
    #face_sam, face_sam_map = sam_func(masked_y, masked_out)
    isam, isam_map = isam_func(y, out, return_maps=True)

    # Convert to numpy for plotting
    sam_map_np = sam_map[0].detach().cpu().numpy()
    isam_map_np = isam_map[0].detach().cpu().numpy()
    #face_sam_map = face_sam_map[0].detach().cpu().numpy()

    # Determine shared color scale limits
    #vmin = min(sam_map_np.min(), isam_map_np.min())
    #vmax = max(sam_map_np.max(), isam_map_np.max())
    vmin=0
    vmax=1.01

    # Create subplots
    fig, axs = plt.subplots(1, 2, figsize=(18, 6))

    # Display SAM map
    im0 = axs[0].imshow(sam_map_np, cmap='magma', vmin=vmin, vmax=vmax)
    axs[0].set_title(f'SAM = {sam:.4f}')

    # Display ISAM map
    #im1 = axs[1].imshow(face_sam_map, cmap='magma', vmin=vmin, vmax=vmax)
    #axs[1].set_title(f'Face SAM = {face_sam:.4f}')

    # Display ISAM map
    im2 = axs[1].imshow(isam_map_np, cmap='magma', vmin=vmin, vmax=vmax)
    axs[1].set_title(f'Intensity SAM = {isam:.4f}')

    # Add a single colorbar for both subplots
    fig.colorbar(im0, ax=axs, orientation='vertical', fraction=0.02, pad=0.04, label='Score Intensity')

    fig.suptitle(model_name)

    plt.savefig(base_path + 'plots/spatial-sam_' + model_name + '.png')

    #plt.tight_layout()
    plt.show()
    

In [16]:
# === Helper Function: Recursively Save to HDF5 ===
def save_dict_to_h5(group: h5py.Group, dictionary: dict):
    """
    Recursively saves a nested dictionary into an HDF5 group.

    Parameters:
    - group (h5py.Group): Current HDF5 group to write into
    - dictionary (dict): Nested dictionary to save
    """
    for key, value in dictionary.items():
        if isinstance(value, dict):
            # Create subgroup and continue recursion
            subgroup = group.create_group(key)
            save_dict_to_h5(subgroup, value)
        elif isinstance(value, (np.ndarray, list)):
            # Save list or numpy array as dataset
            group.create_dataset(key, data=np.array(value))
        else:
            # Save primitive types (str, int, float, bool) as attributes
            group.attrs[key] = value

# === Helper Function: Convert NumPy Arrays to Lists ===
def convert_ndarrays_to_lists(obj: Any) -> Any:
    """
    Recursively convert NumPy arrays in a structure to lists for YAML compatibility.

    Parameters:
    - obj: The object to process (can be dict, list, ndarray, scalar)

    Returns:
    - Object with all ndarrays converted to lists
    """
    if isinstance(obj, dict):
        return {k: convert_ndarrays_to_lists(v) for k, v in obj.items()}
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj

# === Main Function: Save Nested Dictionary ===
def save_nested_dict(
    data: dict,
    filename: str,
    base_path: str = "/media/ana-caznok/SSD-08/recon-segment/inference_results/"
):
    """
    Save a nested dictionary to either a .h5 or .yaml/.yml file.
    Supports lists, NumPy arrays, and primitive types.

    Parameters:
    - data (dict): The nested dictionary to save
    - filename (str): The output file name (must end in .h5, .yaml, or .yml)
    - base_path (str): Directory where the file will be saved
    """
    # Ensure full path by joining base path and filename
    full_path = os.path.join(base_path, filename)

    # Extract the file extension
    ext = os.path.splitext(filename)[1].lower()

    # Save to HDF5
    if ext == ".h5":
        with h5py.File(full_path, "w") as h5file:
            save_dict_to_h5(h5file, data)

    # Save to YAML
    elif ext in [".yaml", ".yml"]:
        data_serializable = convert_ndarrays_to_lists(data)
        with open(full_path, "w") as yaml_file:
            yaml.dump(data_serializable, yaml_file, sort_keys=False)

    # Unsupported format
    else:
        raise ValueError("Unsupported file format. Use .h5 or .yaml/.yml")

### main

In [17]:
base_path = "/media/ana-caznok/SSD-08/recon-segment/"
config_path = "/media/ana-caznok/SSD-08/recon-segment/configs/"

In [45]:
configs = [#'stable_fft2ifft2hsi_1000_none_cluster_xmin.yaml',
           #'fft2hsi_1000_mraessimsam_patch16.yaml', f
           #'msi2hsi_1000_mraessimsam_patch16.yaml', 
           #'msi2mask_wsam_1000.yaml', 
           #'test_feedfoward2.yaml',
           #'test_overlap2.yaml',
           #'test_overlap_mraessimsaunsqueezem.yaml', 
           #'restormer_msi2hsi_mraessimsam.yaml', 
           #'restormer_fft2hsi_mraessimsam.yaml',
           #'restormer_fft2ifft2hsi_mraessimsam.yaml', 
           #'restormer_fft2ifft2hsi_byc.yaml', 
           #'restormer-seg_msi2mask2.yaml' 
           #'restormer_fft2ifft2hsi_noclip.yaml',
           #'restormer_fft2ifft2hsi_interp31.yaml', 
           #'restormer_fft2ifft2hsi_noclip_byc.yaml', 
           #'restormer_fft2ifft2hsi_noclip_minmax.yaml', 
           #'restormer_fft2ifft2hsi_noclip_byimg.yaml'
           #'hscnn_msi2hsi_mraessimsam.yaml', 
           #'hscnn_msi2hsi_mraessimsam.yaml', 
           #'hscnn_fft2hsi_mraessimsam.yaml', = axs[1].imshow(isam_map_np, cmap='magma', vmin=vmin, vmax=vmax)

           #'hscnn_fft2ifft2hsi_mraessimsam.yaml'\
           'unet_msi2hsi.yaml', 
           'unet_fft2ifft2hsi.yaml', 
           'unet_fft2ifft2hsi_interp31.yaml'
           #'restormer_crop_fft2ifft2hsi_interp16.yaml', 
           #'restormer_crop_rgb2hsi.yaml',
           #'unet_crop_fft2ifft2hsi_interp16.yaml',
           #'ntire-restormer_rgb2hsi.yaml', 
           #'ntire-restormer_rgb2fft2ifft2hsi.yaml', 
           #'ntire-unet_rgb2fft2ifft2hsi.yaml',
           #'ntire-unet_rgb2hsi.yaml'
           ]

In [19]:
only_vis = input("is this comparison considers only vis reconstruction?(y/n)")
if only_vis=='y': 
    only_vis = True
else: 
    only_vis = False

datasets = input("does this comparison considers 2 datasets?(y/n)")
if datasets=='y': 
    datasets = True
    from ntire_dataset import NTIRE2022LikeDataset
    ntire_path = '/media/ana-caznok/SSD-08/NTIRE'
else: 
    datasets = False

In [21]:
DEVICE = 'cpu'
BASE_PATH = "/media/ana-caznok/SSD-08/icasp_4090/icasp/data/Link_2/downsampled"
PREPROCESSING = 'downsampled' 
BATCH_SIZE = 1

### only one image

In [ ]:
ssim_dict = {}
error_dict = {}

for config_name in configs: 
    model = []
    config = read_yaml(config_path + config_name)
    name = config_name.split('.')[0]

    TRANSFORM = config['valid']['transform_index']
    MODEL_NAME = config['model']
    need_ifft = bool(config['ifft'])
    print(MODEL_NAME)

    model, resume_file = model_select(config)
    model = model.to(DEVICE)
   
    if 'ntire' in name: 
        val_dataset = NTIRE2022LikeDataset(ntire_path,split='valid', transform=transform_factory(TRANSFORM))

    else: 
        val_dataset = FixedDataset(mode="val",
                                base_path=BASE_PATH,
                                transform=transform_factory(TRANSFORM),
                                preprocessing=PREPROCESSING)
    
    x,y,m = val_dataset[0]
    x = x.to(DEVICE)
    del val_dataset

    if 'mask' in config_name: 
        output,mask = model(x.unsqueeze(0))
        mask = torch.sigmoid(mask) #activating probabilities, will need to change if more then one class
        mask = (mask > 0.5).float() #turning probabilities into binary mask 

        tp = name.split('_')[1].split('2mas')[0]
    
    else: 
        output = model(x.unsqueeze(0))


        if need_ifft: 
            ifft_output_function = inverse_transform_factory(TRANSFORM,output=True)
            output = ifft_output_function(output,m)

        tp = name.split('_')[1].split('2h')[0]
  
    x = x.detach().cpu()

    if isinstance(y,np.ndarray): 
        y = torch.from_numpy(y)
    y = y.to(DEVICE)       

    name = config_name.split('.')[0]
    
    #calc_plot_sam(y.unsqueeze(0),output, m, model_name=name)
    #plot_channel_differences(y, output, channels=[1, 5, 15, 25, 35, 45, 55, 60], model_name=name)
    #plot_output_channels(output,channels=[1, 5, 15, 25, 35, 45, 55, 60],model_name= name)
    plot_spectral_diff(y,output,model_name=name)
    #plot_spectral_input(x,output,y,model_name=name,tp=tp)
    del model, x
    ssim = calc_plot_ssim(y.unsqueeze(0), output, model_name=name)
    ssim_dict[name] = ssim
    
    del y, output
    torch.cuda.empty_cache()
    gc.collect()

plot_all_models(ssim_dict)
plot_grouped_models(ssim_dict, only_vis=only_vis)

### inference for multiple images (statistical tests)

In [46]:
model_dict = {}
error_dict = {}

for config_name in configs: 
    model = []
    val_dataset = []
    
    config = read_yaml(config_path + config_name)
    name = config_name.split('.')[0]

    TRANSFORM = config['valid']['transform_index']
    MODEL_NAME = config['model']
    need_ifft = bool(config['ifft'])
    print(MODEL_NAME)
    print('-------------- NAME:  ' + name + '----------------------')

    model_dict[name] = {'dataset': [],
                        'tp': [], 
                        'fft': need_ifft,
                        'interp': 'interp' in TRANSFORM,
                        'ssim_imgs': [], 
                        'sam_imgs': [],
                        'isam_imgs': [], 
                        'ssim_chann': [], 
                        'error': []}

    model, resume_file = model_select(config)
    model = model.to(DEVICE)
   
    if 'ntire' in name: 
        val_dataset = NTIRE2022LikeDataset(ntire_path,split='valid', transform=transform_factory(TRANSFORM))
        model_dict[name]['dataset'] = 'ntire'

    else: 
        val_dataset = FixedDataset(mode="val",
                                base_path=BASE_PATH,
                                transform=transform_factory(TRANSFORM),
                                preprocessing=PREPROCESSING)
        
        model_dict[name]['dataset'] = 'skin'
    
    for n in range(len(val_dataset)):
        x,y,m = val_dataset[n]
        x = x.to(DEVICE)
        
        output = model(x.unsqueeze(0))

        if need_ifft: 
            ifft_output_function = inverse_transform_factory(TRANSFORM,output=True)
            output = ifft_output_function(output,m)

        model_dict[name]['tp'] = name.split('_')[1].split('2h')[0]
    
        x = x.detach().cpu()

        if isinstance(y,np.ndarray): 
            y = torch.from_numpy(y)

        y = y.to(DEVICE)       
        
        del x,m 

        ssim, channel_ssim = calc_ssim(y.unsqueeze(0),output)
        sam, isam = calc_sam(y.unsqueeze(0),output)
        error = calc_error(y.unsqueeze(0),output)

        model_dict[name]['ssim_imgs'].append(ssim)
        model_dict[name]['ssim_chann'].append(channel_ssim)
        model_dict[name]['sam_imgs'].append(sam)
        model_dict[name]['isam_imgs'].append(isam)
        model_dict[name]['error'].append(error)
        
        del y, output
        
    print('vou deletar hein')
    del val_dataset
    torch.cuda.empty_cache()
    gc.collect()

    save_nested_dict(model_dict,'inf_' + name + '.h5' )

monai-unet_4to61
-------------- NAME:  unet_msi2hsi----------------------
Looking for checkpoint in models/unet_msi2hsi.pth. Exact path only: True.
=> Loading model from checkpoint: 'models/unet_msi2hsi.pth'
Loaded checkpoint from epoch: 49
Resuming from epoch 50/50
Resuming W&B run ID: 1uqrzarr
Selected model monai-unet_4to61: UNet
Number of face masks 30. initialized 30 images val DatasetRaw with preprocessing: downsampled in /media/ana-caznok/SSD-08/icasp_4090/icasp/data/Link_2/downsampled with transform None. Fold: None


/media/ana-caznok/SSD-08/recon-segment/utils/load_ckpt.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(resume_file, map_location=torch.device('c

vou deletar hein
monai-unet_62to122
-------------- NAME:  unet_fft2ifft2hsi----------------------
Looking for checkpoint in models/unet_fft2ifft2hsi.pth. Exact path only: True.
=> Loading model from checkpoint: 'models/unet_fft2ifft2hsi.pth'


/media/ana-caznok/SSD-08/recon-segment/utils/load_ckpt.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(resume_file, map_location=torch.device('c

Loaded checkpoint from epoch: 49
Resuming from epoch 50/50
Resuming W&B run ID: 1lk9r3w4
Selected model monai-unet_62to122: UNet
Number of face masks 30. initialized 30 images val DatasetRaw with preprocessing: downsampled in /media/ana-caznok/SSD-08/icasp_4090/icasp/data/Link_2/downsampled with transform 
0: RGB2Pseudo_Hyp(base_path='/media/ana-caznok/SSD-08/recon-segment/', camera='cie', norm=True, return_torch=True, interpolate=False, final_channels=31)
1: FourierSpectralTransform(norm=None,double RealImag=True ,transform cube=False, device=cuda, shift=True,stack_type=alternate, invert=False ). Fold: None
vou deletar hein
monai-unet_62to122
-------------- NAME:  unet_fft2ifft2hsi_interp31----------------------
Looking for checkpoint in models/unet_fft2ifft2hsi_interp31.pth. Exact path only: True.
=> Loading model from checkpoint: 'models/unet_fft2ifft2hsi_interp31.pth'
Loaded checkpoint from epoch: 49
Resuming from epoch 50/50
Resuming W&B run ID: 2vbr6oys
Selected model monai-unet_

/media/ana-caznok/SSD-08/recon-segment/utils/load_ckpt.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(resume_file, map_location=torch.device('c

vou deletar hein


In [44]:
model_dict[name]

{'dataset': 'skin',
 'tp': 'fft2ifft',
 'fft': True,
 'interp': True,
 'ssim_imgs': [0.95916456,
  0.9582416,
  0.9516537,
  0.958004,
  0.956162,
  0.95035183,
  0.76915556,
  0.82394373,
  0.81780136,
  0.7631975,
  0.7400151,
  0.6741832,
  0.96608263,
  0.9644614,
  0.9661402,
  0.96585816,
  0.9646612,
  0.9642053,
  0.9674092,
  0.9687107,
  0.9630371,
  0.96556646,
  0.9667655,
  0.965924,
  0.9711977,
  0.9676045,
  0.9583546,
  0.9712913,
  0.9646116,
  0.9708364],
 'sam_imgs': [0.06804566830396652,
  0.06483296304941177,
  0.07221725583076477,
  0.06989865005016327,
  0.0675843358039856,
  0.07385209202766418,
  0.1337900161743164,
  0.1273762285709381,
  0.1370134800672531,
  0.14290836453437805,
  0.13184186816215515,
  0.13668763637542725,
  0.05984605848789215,
  0.05616949126124382,
  0.05809061974287033,
  0.06185002624988556,
  0.05703293904662132,
  0.06124163419008255,
  0.07292856276035309,
  0.06942199170589447,
  0.06959228217601776,
  0.07239027321338654,
  0.069

In [ ]:
val_dataset = FixedDataset(mode="val",
                                base_path=BASE_PATH,
                                transform=transform_factory(TRANSFORM),
                                preprocessing=PREPROCESSING)

In [ ]:
def save_nested_dict(data: dict, filename: str):
    """
    Save a nested dictionary to either an .h5 or .yaml file.
    
    Parameters:
    - data (dict): The nested dictionary to save.
    - filename (str): Destination filename. Extension must be either .h5 or .yaml/.yml.
    """
    def recursively_save_to_h5(group, dict_data):
        for key, value in dict_data.items():
            if isinstance(value, dict):
                # Create subgroup and recurse
                subgroup = group.create_group(key)
                recursively_save_to_h5(subgroup, value)
            elif isinstance(value, list):
                # Store lists as datasets
                group.create_dataset(key, data=value)
            else:
                # Store primitive types
                group.attrs[key] = value

    ext = os.path.splitext(filename)[1].lower()
    if ext == ".h5":
        with h5py.File(filename, "w") as h5file:
            recursively_save_to_h5(h5file, data)
    elif ext in [".yaml", ".yml"]:
        with open(filename, "w") as yamlfile:
            yaml.dump(data, yamlfile, sort_keys=False)
    else:
        raise ValueError("Unsupported file format. Use .h5 or .yaml/.yml")

# Example usage:
example_data = {
    'restormer_crop_fft2ifft2hsi_interp16': {
        'dataset': 'skin',
        'tp': 'crop',
        'fft': True,
        'interp': True,
        'ssim_imgs': [
            0.9578136, 0.96006536, 0.94994146, 0.95761985,
            0.959124, 0.9480324, 0.8051259, 0.8077956,
            0.80129194, 0.7991796, 0.7996911, 0.7719804,
            0.9638183, 0.9633773, 0.9629391, 0.96358377,
            0.9630346, 0.9623079, 0.9650345, 0.9660517,
            0.9570745
        ]
    }
}

In [ ]:
model_dict

In [ ]:
y = y.unsqueeze(0) 
mask = m['mask'].unsqueeze(0)
out = output.detach().cpu()
target = replace_zeros_with_small_value(mask*y,0.001)
pred = replace_zeros_with_small_value(mask*out,0.002)
print(pred.shape)
# Compute SAM and Intensity-SAM with corresponding maps
sam_func = SAMScore()

pred, target = pred.squeeze(), target.squeeze()
up = torch.sum((target*pred), dim = 0)   # [w, h]
down1 = torch.sum((target**2), dim = 0).sqrt()
down2 = torch.sum((pred**2), dim = 0).sqrt()

map = torch.arccos(up / (down1 * down2))
score = torch.mean(map[~torch.isnan(map)])
map[torch.isnan(map)] = 0

In [ ]:
np.unique(map.detach().cpu().numpy())

In [ ]:


pred, target = pred.squeeze(), target.squeeze()
up = torch.sum((target*pred), dim = 0)   # [w, h]
down1 = torch.sum((target**2), dim = 0).sqrt()
down2 = torch.sum((pred**2), dim = 0).sqrt()

map = torch.arccos(up / (down1 * down2))
score = torch.mean(map[~torch.isnan(map)])
map[torch.isnan(map)] = 0
